# 82514 · Sesión S18 — Localización con EKF en el almacén

**Bloque 6** · jueves 19 de noviembre de 2026 · 2 h  ·  IQS Universitat Ramon Llull

**Qué hace este cuaderno.** El robot móvil del almacén del curso recorre la ruta del inventario (27,2 m, del muelle por la puerta ancha, el pasillo junto al muro y de vuelta por la estrecha) y se localiza con un EKF: la odometría como composición en SE(2) y un sensor de distancia y rumbo a diez balizas, con sus jacobianos comprobados por diferencias finitas. Se compara con la odometría sola, se juzga con tres métricas (el error, la consistencia de la elipse y la innovación) y se provocan sus fallos: tramos sin balizas, la baliza lejana, una asociación equivocada y la divergencia.

**Se apoya en:** Corke (2023), cap. 6; Thrun, Burgard y Fox (2005), caps. 3 y 5; documentación de numpy.

**Cómo usarlo en clase.** Sigue el guion de la sesión S18 en las diapositivas del bloque 6. Ejecuta la primera celda una sola vez al empezar; en Colab tarda unos segundos. Las celdas marcadas **Ejercicio** son para que los trabajen los estudiantes: las soluciones están al final del cuaderno.

---

In [ ]:
# Ejecutar una sola vez al abrir el cuaderno. En Colab tarda unos segundos: no instala nada.
import sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

VERSIONES = {'numpy': '2.4.6', 'matplotlib': '3.10.0'}   # las de todo el bloque 6 (las de Colab al prepararlo)
EN_COLAB = 'google.colab' in sys.modules
np.set_printoptions(precision=3, suppress=True)
plt.rcParams['figure.figsize'] = (10, 3.4)
plt.rcParams['axes.grid'] = True
IQS_AZUL, IQS_VERDE, GRIS, AMBAR, ROJO = '#1B2A80', '#1FA355', '#8A8FA8', '#B9770E', '#C0392B'
COMA = FuncFormatter(lambda v, p: f'{v:g}'.replace('.', ',').replace('-', '−'))   # ejes con coma decimal

def coma(x, n=2):
    """Un número con coma decimal y signo menos, para imprimir."""
    return f'{x:.{n}f}'.replace('.', ',').replace('-', '−')

def cien(v, n=2):
    """Un número pequeño o grande con coma decimal y potencia de diez: 1,40·10⁻¹⁰."""
    m, e = f'{v:.{n}e}'.split('e')
    return m.replace('.', ',').replace('-', '−') + '·10' + str(int(e)).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))

def ejes(*axs):
    for ax in axs:
        ax.xaxis.set_major_formatter(COMA); ax.yaxis.set_major_formatter(COMA)

print('numpy', np.__version__, '· matplotlib', matplotlib.__version__, '· en Colab:', EN_COLAB)
if not np.__version__.startswith('2.4'):
    print('AVISO: este cuaderno se probó con numpy', VERSIONES['numpy'], '; las cifras con semilla pueden cambiar un poco.')

> **Aviso para Colab.** Ábrelo al acabar el taller S18a y ejecuta la primera celda: no instala nada y tarda unos segundos. No depende del S18a: define sus propios datos.
>
> **En clase (ocho minutos):** las secciones 1 y 2, sin detenerte, y de la 3 a la 7, ejecutándolas y leyendo lo que imprimen, con una pregunta en cada una (en la 6, añadiendo distancias); se ponen en común al acabar.
>
> **En casa:** la sección 8 y los ejercicios 1 a 3; las soluciones están al final.
>
> **No se sube:** este cuaderno no tiene tarea en Moodle.

## 1. El almacén, el robot y las balizas

*En clase, ejecuta estas celdas sin detenerte. Diapositivas 46, 58 y 59.*

**El almacén** es el que planificaréis en S22, el jueves 3 de diciembre: una rejilla de 90 × 130 celdas de 10 cm (13 × 9 m), con dos naves, un muro con una puerta ancha (2,0 m) y una estrecha (0,8 m), y estanterías. El muelle está en (0,6; 0,6) y el puesto de picking, en (8,3; 8,2).

**El robot** es el móvil del bloque: diferencial, con la rueda de S6, a 0,5 m/s, con la odometría a 10 Hz (pasos de 5 cm). Su odometría se equivoca 6 cm y 0,9° por metro: por paso, σ_d = 13,4 mm y σ_θ = 0,2°. Es **Q**, la covarianza del ruido del proceso.

**El sensor** mide a cada baliza visible (hasta 8 m, si nada la tapa) la distancia, con σ = 5 cm, y el rumbo, con σ = 1°: es **R**, la covarianza del ruido de la medida. Da un barrido por segundo, cada 10 pasos. Q del proceso y R de la medida, como en python-control; Thrun et al. (2005) las llaman al revés y Corke (2023), V y W.

La celda siguiente es el simulador entero: el mapa, los dos modelos con sus jacobianos, lo que leen los sensores (`simula`) y el filtro (`ekf`). La ruta es la verdad; la odometría y las medidas llevan su ruido.

In [ ]:
# El almacen del curso y el robot movil del bloque 6 (S18 y S19). Es la celda del simulador del cuaderno S18b:
# las cifras de las diapositivas salen de este mismo codigo.
#  - El almacen de S22: rejilla de 90 x 130 celdas de 10 cm (13 x 9 m), dos naves, un muro con una puerta
#    ancha (2,0 m) y una estrecha (0,8 m), y estanterias. El muelle en (0,6; 0,6) y el puesto de picking en (8,3; 8,2).
#  - El robot: diferencial, con la rueda de S6; a 0,5 m/s, con la odometria a 10 Hz (pasos de 5 cm).
#    Ruido de la odometria: 6 cm y 0,9° por metro recorrido (por paso, 13,4 mm y 0,2°).
#  - Diez balizas reflectantes y un sensor de distancia y rumbo: 5 cm y 1°, alcance de 8 m con linea de vista,
#    un barrido por segundo (cada 10 pasos de odometria, 0,5 m).
import numpy as np

D2R = np.pi / 180
RES, FILAS, COLS = 0.10, 90, 130            # celdas de 10 cm: 13 m en x, 9 m en y


def mapa_almacen():
    """La rejilla de ocupacion del almacen de S22 (True = ocupado). Fila = y/0,1; columna = x/0,1."""
    occ = np.zeros((FILAS, COLS), bool)
    occ[0, :] = occ[-1, :] = True              # paredes exteriores
    occ[:, 0] = occ[:, -1] = True
    occ[42:47, :] = True                       # el muro central, de y = 4,2 a 4,7 m
    occ[42:47, 8:28] = False                   #   la puerta ancha, de x = 0,8 a 2,8 m
    occ[42:47, 92:100] = False                 #   la puerta estrecha, de x = 9,2 a 10,0 m
    occ[12:36, 40:47] = True                   # estanterias de la nave de abajo
    occ[12:36, 64:71] = True
    occ[58:78, 34:41] = True                   # estanterias de la nave de arriba
    occ[58:78, 58:65] = True
    occ[58:78, 82:89] = True
    return occ


OCC = mapa_almacen()
MUELLE, PICKING = np.array([0.6, 0.6]), np.array([8.3, 8.2])
BALIZAS = np.array([[0.1, 2.0], [3.0, 0.1], [8.0, 0.1], [12.9, 2.0], [5.5, 4.15],      # nave de abajo
                    [0.1, 7.0], [4.7, 8.9], [10.5, 8.9], [12.9, 6.5], [7.0, 4.75]])    # nave de arriba
RUTA = np.array([[0.6, 0.6], [1.8, 0.6], [1.8, 5.2], [9.6, 5.2], [9.6, 0.6], [0.6, 0.6]])   # la ruta del inventario

V, HZ = 0.5, 10                              # 0,5 m/s; odometria a 10 Hz
DD = V / HZ                                  # 5 cm por paso
SIG_D, SIG_T = 0.06 / np.sqrt(20), 0.2 * D2R # por paso: 13,4 mm y 0,2° (6 cm y 0,9° por metro)
Q = np.diag([SIG_D**2, SIG_T**2])            # Q: la covarianza del ruido de proceso (la odometria)
SIG_R, SIG_PHI, ALCANCE = 0.05, 1.0 * D2R, 8.0
R = np.diag([SIG_R**2, SIG_PHI**2])          # R: la covarianza del ruido de medida (el sensor de balizas)
CADA = 10                                    # un barrido de balizas cada 10 pasos: una vez por segundo
K95 = 2.4477                                 # en el plano, la elipse de 2,45σ encierra el 95 %


def envuelve(a):
    """Un angulo en (−π, π]: sin esto, la innovacion del rumbo salta 360°."""
    return (a + np.pi) % (2 * np.pi) - np.pi


def paso(x, d):
    """El modelo de movimiento: la pose x = (x; y; θ) compuesta con la odometria d = (δ_d; δ_θ) en SE(2),
    como en S8: avanzar δ_d en la direccion del rumbo y despues girar δ_θ."""
    return np.array([x[0] + d[0] * np.cos(x[2]), x[1] + d[0] * np.sin(x[2]), envuelve(x[2] + d[1])])


def jac_paso(x, d):
    """Sus jacobianos: F_x respecto a la pose (3 x 3) y F_v respecto a la odometria (3 x 2)."""
    c, s = np.cos(x[2]), np.sin(x[2])
    Fx = np.array([[1.0, 0.0, -d[0] * s], [0.0, 1.0, d[0] * c], [0.0, 0.0, 1.0]])
    Fv = np.array([[c, 0.0], [s, 0.0], [0.0, 1.0]])
    return Fx, Fv


def h(x, b):
    """El modelo de medida: distancia y rumbo a la baliza b desde la pose x."""
    dx, dy = b[0] - x[0], b[1] - x[1]
    return np.array([np.hypot(dx, dy), envuelve(np.arctan2(dy, dx) - x[2])])


def jac_h(x, b):
    """Su jacobiano H_x respecto a la pose (2 x 3)."""
    dx, dy = b[0] - x[0], b[1] - x[1]
    r2 = dx * dx + dy * dy
    r = np.sqrt(r2)
    return np.array([[-dx / r, -dy / r, 0.0], [dy / r2, -dx / r2, -1.0]])


def libre_vista(a, b):
    """¿Ve el sensor la baliza b desde a? Sin celdas ocupadas en medio (la baliza esta en la pared)."""
    n = int(np.ceil(np.linalg.norm(b - a) / 0.02))
    p = a + np.linspace(0, 1, n)[:, None] * (b - a)
    cc = np.clip((p[:, 0] / RES).astype(int), 0, COLS - 1)
    ff = np.clip((p[:, 1] / RES).astype(int), 0, FILAS - 1)
    return not OCC[ff, cc][:-6].any()


def ruta_densa():
    """La ruta del inventario, en pasos de 5 cm."""
    pts = [RUTA[0]]
    for a, b in zip(RUTA[:-1], RUTA[1:]):
        n = int(round(np.linalg.norm(b - a) / DD))
        pts += [a + (b - a) * k / n for k in range(1, n + 1)]
    return np.array(pts)


def mandos(rd):
    """La odometria verdadera de cada paso: el avance y el giro que llevan de un punto de la ruta al siguiente."""
    d, th = [], 0.0
    for a, b in zip(rd[:-1], rd[1:]):
        thn = np.arctan2(b[1] - a[1], b[0] - a[0])
        d.append((np.linalg.norm(b - a), envuelve(thn - th)))
        th = thn
    return np.array(d)


RD = ruta_densa()
U = mandos(RD)                               # 544 pasos, 27,2 m
VERDAD = [np.array([RD[0, 0], RD[0, 1], 0.0])]
for d in U:
    VERDAD.append(paso(VERDAD[-1], d))
VERDAD = np.array(VERDAD)
VISIBLES = [[j for j, b in enumerate(BALIZAS) if np.linalg.norm(b - p[:2]) < ALCANCE and libre_vista(p[:2], b)]
            for p in VERDAD]


def simula(semilla=1):
    """Lo que leen los sensores en una pasada por la ruta: la odometria de cada paso y, una vez por segundo,
    la distancia y el rumbo de cada baliza visible (con su numero: la asociacion se supone conocida)."""
    rng = np.random.default_rng(semilla)
    odo = U + rng.normal(0, [SIG_D, SIG_T], size=U.shape)
    med = {}
    for k in range(CADA, len(VERDAD), CADA):
        med[k] = [(j, h(VERDAD[k], BALIZAS[j]) + rng.normal(0, [SIG_R, SIG_PHI])) for j in VISIBLES[k]]
    return odo, med


SIGMA0 = np.diag([0.02**2, 0.02**2, (1.0 * D2R)**2])   # al salir del muelle: 2 cm y 1°


def ekf(odo, med, balizas=range(10), corregir=True, x0=None, Sigma0=SIGMA0, cambia=None, puerta=None):
    """El EKF de localizacion: predice con la odometria en cada paso y corrige con cada baliza vista.
    balizas: las que estan en el mapa. corregir=False: a estima, solo prediccion.
    cambia = (k, j_real, j_usada): en el barrido k, la medida de la baliza j_real se atribuye a j_usada.
    puerta: si se da (por ejemplo 9,21), se descartan las medidas con d² = νᵀ·S⁻¹·ν por encima.
    Devuelve las poses, las covarianzas, las d² de cada medida y los saltos de la correccion."""
    x = VERDAD[0].copy() if x0 is None else np.array(x0, float)
    S_ = Sigma0.copy()
    X, P, D2, SALTOS = [x.copy()], [S_.copy()], [], []
    for k, d in enumerate(odo, start=1):
        Fx, Fv = jac_paso(x, d)                       # prediccion
        x = paso(x, d)
        S_ = Fx @ S_ @ Fx.T + Fv @ Q @ Fv.T
        if corregir and k in med:                     # correccion, una baliza cada vez
            x_antes = x.copy()
            for j, z in med[k]:
                if j not in balizas:
                    continue
                jm = cambia[2] if (cambia and k == cambia[0] and j == cambia[1]) else j
                Hx = jac_h(x, BALIZAS[jm])
                nu = z - h(x, BALIZAS[jm])
                nu[1] = envuelve(nu[1])               # la innovacion
                Sm = Hx @ S_ @ Hx.T + R                # su covarianza
                d2 = float(nu @ np.linalg.solve(Sm, nu))   # Mahalanobis
                D2.append((k, j, jm, d2))
                if puerta is not None and d2 > puerta:
                    continue
                L = S_ @ Hx.T @ np.linalg.inv(Sm)      # la ganancia de Kalman
                x = x + L @ nu
                x[2] = envuelve(x[2])
                S_ = (np.eye(3) - L @ Hx) @ S_
            SALTOS.append((k, np.linalg.norm(x[:2] - x_antes[:2])))
        X.append(x.copy()); P.append(S_.copy())
    return np.array(X), np.array(P), D2, SALTOS


def metricas(X, P):
    """Error de posicion frente a la verdad: RMSE, maximo y final; y la consistencia: el % de pasos con el
    error dentro de la elipse del 95 % (2,45σ) de su propia Σ."""
    e = X[:, :2] - VERDAD[:, :2]
    n = np.linalg.norm(e, axis=1)
    dentro = [ei @ np.linalg.solve(Pi[:2, :2], ei) <= K95**2 for ei, Pi in zip(e[1:], P[1:])]
    return dict(rmse=np.sqrt((n**2).mean()), maximo=n.max(), final=n[-1], dentro=100 * np.mean(dentro), err=n)


def elipse(Sig, n=K95, m=60):
    """Los puntos de la elipse de n·σ de una covarianza 2 x 2 (sus semiejes, n·√ de los autovalores)."""
    w, v = np.linalg.eigh(Sig)
    t = np.linspace(0, 2 * np.pi, m)
    return (v @ (n * np.sqrt(np.maximum(w, 0))[:, None] * np.array([np.cos(t), np.sin(t)]))).T

In [ ]:
L_ruta = np.sum(np.linalg.norm(np.diff(RUTA, axis=0), axis=1))
nv = [len(v) for v in VISIBLES]
print(f'La ruta del inventario: {coma(L_ruta,1)} m en {len(U)} pasos de 5 cm ; balizas a la vista: de {min(nv)} a {max(nv)}')

def dibuja_mapa(ax):
    ax.imshow(OCC, origin='lower', extent=(0, 13, 0, 9), cmap='Greys', vmin=0, vmax=1.6)
    ax.plot(BALIZAS[:, 0], BALIZAS[:, 1], '*', color=IQS_VERDE, ms=12)
    ax.plot(*MUELLE, 's', color=AMBAR); ax.plot(*PICKING, 's', color=AMBAR)
    ax.set_aspect('equal'); ejes(ax)

fig, ax = plt.subplots(figsize=(7, 5))
dibuja_mapa(ax); ax.plot(VERDAD[:, 0], VERDAD[:, 1], color='black', lw=2)
ax.set_title('El almacén, la ruta del inventario y las diez balizas'); plt.show()

## 2. Los modelos y sus jacobianos, comprobados

*En clase, sin detenerte. Diapositivas 57 a 59.*

El EKF cambia cada función por su tangente en la estimación: los **jacobianos**, como el J de S11. Si un jacobiano está mal, el filtro diverge sin decir por qué; por eso se comprueban con diferencias finitas, como en S11.

In [ ]:
def numerico(f, x, h=1e-6):
    """Jacobiano por diferencias centradas, columna a columna (con el ángulo envuelto)."""
    cols = []
    for j in range(len(x)):
        e = np.zeros(len(x)); e[j] = h
        d = f(x + e) - f(x - e)
        if len(d) == 3: d[2] = envuelve(d[2])
        if len(d) == 2: d[1] = envuelve(d[1])
        cols.append(d / (2 * h))
    return np.array(cols).T

xp, dp, b = np.array([1.8, 2.0, 0.6]), np.array([0.05, 0.004]), BALIZAS[9]
Fx, Fv = jac_paso(xp, dp)
print('F_x, diferencia máxima:', cien(np.abs(numerico(lambda x: paso(x, dp), xp) - Fx).max()))
print('F_v, diferencia máxima:', cien(np.abs(numerico(lambda d: paso(xp, d), dp) - Fv).max()))
xr = np.array([6.0, 5.2, 0.0])
print('H_x, diferencia máxima:', cien(np.abs(numerico(lambda x: h(x, b), xr) - jac_h(xr, b)).max()))
z = h(xr, b)
print(f'Desde (6,0; 5,2; 0°), la baliza (7,0; 4,75): r = {coma(z[0])} m, φ = {coma(np.degrees(z[1]),1)}°')
print('H_x =', ' ; '.join('(' + ', '.join(coma(v, 3) for v in f) + ')' for f in jac_h(xr, b)))

## 3. A estima y con el EKF, por la ruta

*En clase. Diapositivas 58 y 60.*

`ekf(odo, med, corregir=False)` solo predice: es la navegación a estima. Con `corregir=True`, el ciclo completo: predecir con la odometría en cada paso y, en cada barrido, corregir con cada baliza vista (la **innovación** ν = z − h(x̂⁻), su covarianza S, la ganancia L). Se supone que se sabe qué baliza es cada medida: la asociación, conocida.

**En clase.** ¿Cuánto se equivoca cada uno, a estima y con el EKF, y cómo es su elipse?

In [ ]:
SEMILLA = 18                        # probad otras: las cifras cambian un poco
odo, med = simula(SEMILLA)
Xd, Pd, _, _ = ekf(odo, med, corregir=False)   # a estima
X, P, D2, SALTOS = ekf(odo, med)                # EKF
md_, m = metricas(Xd, Pd), metricas(X, P)
print(f'A estima: error medio (RMSE) {coma(md_["rmse"]*100,1)} cm, máximo {coma(md_["maximo"]*100,1)}, final {coma(md_["final"]*100,1)}')
print(f'EKF:      error medio (RMSE) {coma(m["rmse"]*100,1)} cm, máximo {coma(m["maximo"]*100,1)}, final {coma(m["final"]*100,1)}')
s = np.array([sl for _, sl in SALTOS])
print(f'Al corregir, la estimación salta {coma(s.mean()*100,1)} cm de media y {coma(s.max()*100,1)} como mucho')

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={'width_ratios': [1.3, 1]})
dibuja_mapa(a1)
a1.plot(VERDAD[:, 0], VERDAD[:, 1], color='black', lw=2, label='verdad')
a1.plot(Xd[:, 0], Xd[:, 1], '--', color=ROJO, label='a estima')
a1.plot(X[:, 0], X[:, 1], color=IQS_AZUL, label='EKF')
for k in range(0, len(X), 90):
    e = elipse(Pd[k][:2, :2]) + Xd[k, :2]; a1.plot(e[:, 0], e[:, 1], color=ROJO, lw=1)
a1.legend(fontsize=8)
t = np.arange(len(X)) / 10
semieje = lambda Pk: np.sqrt(np.linalg.eigvalsh(Pk[:2, :2]).max()) * 100
a2.semilogy(t, [semieje(p) for p in Pd], color=ROJO, label='a estima')
a2.semilogy(t, [semieje(p) for p in P], color=IQS_AZUL, label='EKF')
a2.set_xlabel('t (s)'); a2.set_ylabel('semieje mayor de σ (cm)'); a2.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 4. Cómo se juzga un filtro: las métricas

*En clase. Diapositiva 62.*

- **El error:** el RMSE, la raíz de la media del error de posición al cuadrado, frente a la verdad, y frente a la odometría sola.
- **La elipse honesta:** el porcentaje del tiempo en que el error cae dentro de su propia elipse de 2,45σ, la del 95 % en el plano. Si sale mucho menos del 95 %, el filtro se cree mejor de lo que es; si sale el 100 % con una elipse enorme, es honesto pero inútil.
- **La innovación avisa:** d² = νᵀ·S⁻¹·ν, el cuadrado de la sorpresa de cada medida, medida en desviaciones típicas de lo que se esperaba (la distancia de Mahalanobis, al cuadrado). De media vale 2, uno por cada número medido (la distancia y el rumbo); por encima de 9,21 solo cae el 1 % de las medidas buenas.

Estas tres son las métricas de la entrega 3.

**En clase.** Con el error medio y el % dentro de 2,45σ: ¿es honesta la elipse?

In [ ]:
d2 = np.array([d[3] for d in D2])
print(f'{"":10s}{"RMSE":>9s}{"máximo":>9s}{"final":>9s}{"dentro de 2,45σ":>17s}')
for nombre, mm in (('a estima', md_), ('EKF', m)):
    print(f'{nombre:10s}{coma(mm["rmse"]*100,1):>7s} cm{coma(mm["maximo"]*100,1):>7s} cm{coma(mm["final"]*100,1):>7s} cm{coma(mm["dentro"],1):>15s} %')
print(f'd² de las {len(d2)} medidas: media {coma(d2.mean())} ; por encima de 5,99, el {coma(100*(d2>5.99).mean(),1)} % ; de 9,21, el {coma(100*(d2>9.21).mean(),1)} %')

## 5. Quitar balizas

*En clase. Diapositiva 62.*

La celda ya deja en el mapa solo las cinco balizas de la nave de abajo (`BALIZAS_EN_MAPA = range(5)`; con las diez, `range(10)`): ¿qué pasa en el pasillo de la nave de arriba?

In [ ]:
BALIZAS_EN_MAPA = range(5)          # todas: range(10); solo la nave de abajo: range(5)

Xb, Pb, _, SALb = ekf(odo, med, balizas=list(BALIZAS_EN_MAPA))
mb = metricas(Xb, Pb)
sin = [k for k in med if not any(j in BALIZAS_EN_MAPA for j, _ in med[k])]
print(f'Barridos sin ninguna baliza: {len(sin)}' + (f' (de t = {coma(sin[0]/10, 0)} a {coma(sin[-1]/10, 0)} s)' if sin else ''))
print(f'RMSE {coma(mb["rmse"]*100,1)} cm, máximo {coma(mb["maximo"]*100,1)} cm, dentro de 2,45σ el {coma(mb["dentro"],1)} %')
print(f'El semieje mayor llega a {coma(max(semieje(p) for p in Pb),1)} cm ; el salto más grande, {coma(max(s_ for _, s_ in SALb)*100,1)} cm')
t = np.arange(len(Xb)) / 10
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t, mb['err'] * 100, color=IQS_AZUL, label='error')
ax.plot(t, [2.4477 * semieje(p) for p in Pb], '--', color=GRIS, label='2,45σ (semieje mayor)')
ax.set_xlabel('t (s)'); ax.set_ylabel('cm'); ax.legend(fontsize=8); ejes(ax); plt.show()

## 6. La baliza lejana

*En clase. Diapositiva 59.*

El robot cree saber su pose con σ = 10 cm a lo largo, 10 cm de lado y 2° de rumbo, y lee **una** baliza que tiene delante, a la distancia `D`. Un grado de rumbo son r·π/180 de lado: 3,5 cm a 2 m y 14 cm a 8 m; pero ∂φ/∂θ = −1 a cualquier distancia. A 2 y a 8 m es la tabla de la diapositiva 59.

**En clase.** Añade distancias a `DISTANCIAS`: ¿desde dónde la lectura apenas sitúa de lado (deja más de 9 cm, casi los 10 de antes)? ¿Y el rumbo?

In [ ]:
DISTANCIAS = (2.0, 8.0)              # en m: probad 3, 4, 5, 6...
P0 = np.diag([0.10**2, 0.10**2, np.radians(2.0)**2])
for D in DISTANCIAS:
    Hx = jac_h(np.zeros(3), np.array([D, 0.0]))
    S_ = Hx @ P0 @ Hx.T + R
    Lg = P0 @ Hx.T @ np.linalg.inv(S_)
    Pp = (np.eye(3) - Lg @ Hx) @ P0
    print(f'a {coma(D, 1)} m: σ a lo largo {coma(np.sqrt(Pp[0,0])*100,1)} cm, de lado {coma(np.sqrt(Pp[1,1])*100,1)} cm,'
          f' rumbo {coma(np.degrees(np.sqrt(Pp[2,2])),2)}° ; 1° de rumbo son {coma(D*np.pi/180*100,1)} cm de lado')

## 7. Una asociación equivocada, y la puerta de Mahalanobis

*En clase. Diapositiva 62.*

En el barrido de t = 20 s (x = 6,0 m del pasillo), la medida de la baliza del muro de arriba, (7,0; 4,75), se atribuye a la del muro de abajo, (5,5; 4,15). ¿Cuánto salta la estimación, y qué dice d²? Después, con la **puerta**: se descartan las medidas con d² por encima de 9,21.

In [ ]:
CAMBIO = (200, 9, 4)                 # (barrido, baliza de verdad, baliza a la que se atribuye)

for puerta in (None, 9.21):
    Xw, Pw, D2w, _ = ekf(odo, med, cambia=CAMBIO, puerta=puerta)
    mw = metricas(Xw, Pw)
    d2w = [d[3] for d in D2w if d[0] == CAMBIO[0] and d[1] == CAMBIO[1]][0]
    etiqueta = 'sin puerta' if puerta is None else f'con la puerta en {coma(puerta)}'
    print(f'{etiqueta}: d² de la equivocada = {coma(d2w,0)} ; error justo después {coma(mw["err"][CAMBIO[0]]*100,1)} cm ;'
          f' RMSE {coma(mw["rmse"]*100,1)} cm ; dentro de 2,45σ el {coma(mw["dentro"],1)} %')

## 8. Romper el EKF (en casa)

*Diapositiva 57.*

El EKF linealiza en su estimación. Si está lejos de la verdad, la tangente se equivoca, y la covarianza sigue diciendo «estoy seguro»: con no linealidad fuerte o incertidumbre grande, **puede divergir sin avisar**. El robot sale del muelle con el rumbo desconocido (90° de error, y el filtro lo sabe: σ_θ = 90°), y solo ve una baliza, la (3,0; 0,1).

In [ ]:
x0 = VERDAD[0] + np.array([0, 0, np.radians(90)])
S0 = np.diag([0.02**2, 0.02**2, np.radians(90)**2])
for nombre, bal in (('una baliza', [1]), ('diez balizas', range(10))):
    Xr, Pr, D2r, _ = ekf(odo, med, balizas=list(bal), x0=x0, Sigma0=S0)
    mr = metricas(Xr, Pr)
    sf = np.sqrt(Pr[-1, 0, 0] + Pr[-1, 1, 1])
    print(f'{nombre}: error máximo {coma(mr["maximo"],2)} m, final {coma(mr["final"]*100,1)} cm con σ = {coma(sf*100,1)} cm'
          f' ({coma(mr["final"]/sf,1)}σ) ; últimas d²: {", ".join(coma(d[3],1) for d in D2r[-5:])}')

## Ejercicios (en casa)

### Ejercicio 1 · Un sensor de rumbo peor

Cambia el sensor de rumbo de 1° a 5° (`SIG_PHI` y `R`), vuelve a simular y a filtrar. ¿Qué empeora más, la posición o el rumbo? ¿Sigue siendo honesta la elipse?

In [ ]:
# Ejercicio 1
# SIG_PHI = np.radians(5.0); R = np.diag([SIG_R**2, SIG_PHI**2])
# o5, m5 = simula(18); X5, P5, _, _ = ekf(o5, m5); r5 = metricas(X5, P5)
# print(coma(r5['rmse']*100,1), coma(r5['dentro'],1), coma(np.degrees(np.sqrt(P5[1:,2,2]).mean()),2))
# SIG_PHI = np.radians(1.0); R = np.diag([SIG_R**2, SIG_PHI**2])      # dejarlo como estaba

### Ejercicio 2 · ¿Cuántas balizas hacen falta?

Quita balizas del mapa, de una en una, quitando cada vez la que menos empeora el RMSE. ¿Con cuántas sigue por debajo de 5 cm? ¿Cuáles quedan, y por qué esas?

In [ ]:
# Ejercicio 2
# quedan = list(range(10))
# while len(quedan) > 1:
#     prueba = []
#     for j in quedan:
#         Xq, Pq, _, _ = ekf(odo, med, balizas=[i for i in quedan if i != j])
#         prueba.append((metricas(Xq, Pq)['rmse'], j))
#     rm, j = min(prueba); quedan.remove(j)
#     print(len(quedan), 'balizas:', ' '.join(f'({coma(BALIZAS[i][0],1)}; {coma(BALIZAS[i][1],2)})' for i in quedan), '· RMSE', coma(rm*100,1), 'cm')

### Ejercicio 3 · El umbral de la puerta

Con la puerta de Mahalanobis en 9,21, en 5,99 y en 3, ¿cuántas medidas buenas se descartan, y qué pasa con el RMSE y con el porcentaje dentro de 2,45σ? ¿Por qué no se pone el umbral muy bajo?

In [ ]:
# Ejercicio 3
# for u in (9.21, 5.99, 3.0):
#     Xg, Pg, Dg, _ = ekf(odo, med, puerta=u); mg = metricas(Xg, Pg)
#     print('umbral', coma(u), ':', sum(d[3] > u for d in Dg), 'de', len(Dg), 'descartadas ; RMSE', coma(mg['rmse']*100,1), 'cm ; dentro', coma(mg['dentro'],1), '%')

---

## Soluciones

**Ejercicio 1.** Con 5° de rumbo, el RMSE de posición pasa de 4,0 a 5,1 cm y el máximo, de 12,7 a 12,8: la posición casi no se entera, porque la sujetan las distancias, que siguen con 5 cm. El rumbo sí: su σ media pasa de 0,63° a 1,15°. La elipse sigue honesta (el 93,2 % dentro de 2,45σ) porque R dice la verdad sobre el sensor. Si cambiaras el sensor de verdad y no R, el filtro se creería mejor de lo que es.

**Ejercicio 2.** Quitando cada vez la que menos empeora: con 9, 8 y 7 balizas el RMSE se queda en 3,8-3,9 cm; con 6, en 4,0; con 5, en 4,2; con 4 —(0,1; 2,0), (3,0; 0,1), (10,5; 8,9) y (7,0; 4,75)—, en 4,5; con 2 —(3,0; 0,1) y (7,0; 4,75)—, en 6,8; y con una sola, en 15,8. Quedan las que se ven desde más tramos de la ruta, una en cada nave y alguna junto al muro: lo que importa no es cuántas hay, sino que siempre haya alguna a la vista, y en direcciones distintas.

**Ejercicio 3.** Con 9,21 se descarta 1 medida buena de 182 (RMSE 4,0 cm, 95,0 % dentro); con 5,99, 11 (4,3 cm, 93,4 %); con 3, 53 (6,6 cm, 80,3 %). Un umbral bajo tira medidas buenas, el filtro corrige menos y deja de ser consistente. La puerta está para las medidas imposibles, como la asociación equivocada (d² = 2580), no para las que sorprenden un poco.

---

## Para llevarse de esta sesión

El EKF de localización es Kalman con los dos modelos del robot: la odometría como composición en SE(2) y la distancia y el rumbo a las balizas, linealizados con sus jacobianos (comprobados por diferencias finitas). Con diez balizas, el error medio baja de 27 cm a estima a 4 cm.

Un filtro se juzga con tres números: el error frente a la verdad y frente a la odometría sola; el porcentaje del error dentro de su elipse de 2,45σ, que tiene que rondar el 95 %; y la innovación, d², que avisa de lo imposible. Y se le reconocen los fallos: los tramos sin medidas, la baliza lejana, la asociación equivocada y la divergencia sin aviso.

El lunes 23, en S19: qué hacer cuando la creencia tiene más de un pico, y cuando tampoco hay mapa.

*Cuaderno del curso 82514 Mecatrónica y Robótica · IQS Universitat Ramon Llull · curso 2026/27*

*© Guillermo Reyes Carmenaty · Publicado bajo [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es): puedes usarlo, adaptarlo y redistribuirlo, incluso con fines comerciales, siempre que reconozcas la autoría.*